In [0]:
from pyspark.sql import functions as F

# Leemos la tabla silver
df_s = spark.table("workspace.silver.superstore")

# Agregado 1: Ventas por categoría y subcategoría
gold_categoria = (
    df_s.groupBy("category", "sub_category")
    .agg(
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.sum("quantity").alias("total_quantity"),
        F.round(F.avg("discount"), 4).alias("avg_discount"),
        F.round(F.sum("profit") / F.sum("sales") * 100, 2).alias("profit_margin_pct"),
        F.countDistinct("order_id").alias("num_orders")
    )
    .withColumn("fec_informacion", F.current_timestamp())
)
gold_categoria.display()


In [0]:
# Agregado 2: Ventas por región y segmento de cliente
gold_region_segmento = (
    df_s.groupBy("region", "segment")
    .agg(
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("customer_id").alias("num_customers"),
        F.countDistinct("order_id").alias("num_orders")
    )
    .withColumn("fec_informacion", F.current_timestamp())
)
gold_region_segmento.display()

In [0]:
# Agregado 3: Performance por cliente (top customers)
gold_clientes = (
    df_s.groupBy("customer_id", "customer_name", "segment", "region")
    .agg(
        F.round(F.sum("sales"), 2).alias("total_sales"),
        F.round(F.sum("profit"), 2).alias("total_profit"),
        F.sum("quantity").alias("total_quantity"),
        F.countDistinct("order_id").alias("num_orders"),
        F.round(F.avg("discount"), 4).alias("avg_discount")
    )
    .withColumn("customer_rank",
        F.dense_rank().over(
            __import__("pyspark.sql.window", fromlist=["Window"])
            .Window.orderBy(F.desc("total_sales"))
        )
    )
    .withColumn("fec_informacion", F.current_timestamp())
)
gold_clientes.display()

In [0]:
# guardamos tablas gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

gold_categoria.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.superstore_categoria")

gold_region_segmento.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.superstore_region_segmento")

gold_clientes.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.superstore_clientes")

print(f"   - workspace.gold.superstore_categoria:       {gold_categoria.count()} filas")
print(f"   - workspace.gold.superstore_region_segmento: {gold_region_segmento.count()} filas")
print(f"   - workspace.gold.superstore_clientes:        {gold_clientes.count()} filas")